# 03 SLEAP 跟踪与导出

本 Notebook 对应 **scripts/sleap**：SLEAP 模型推理、导出

## 流程概览

1. **视频推理**：使用训练好的 Centroid + Instance 模型对视频进行预测（或使用 `batch_track.py` 批量处理）
2. **导出数据**：将 `.slp` 导出为 CSV / H5（`export_tracks.py`）
3. **叠加视频**：将 SLEAP 坐标叠加到原视频并导出（`sleap render` 或 Python 绘制）
。

## 点标记，模型训练

sleap-label

## 1. 视频推理

只需在 **VIDEO_LIST** 中填写 **top 视角视频** 路径。运行下方 cell 会：
1. 对每个 top 视频用 **Top 模型** 跟踪；
2. **自动查找对应的 side 视频**（PLAYER1→PLAYER2/PLAYER3，或 cam0→cam1/cam2），并用 **Side 模型** 跟踪（找到几个就跟踪几个）。

找不到 side 视频时会跳过、不报错（可关闭 `SKIP_MISSING_SIDE` 改为报错）。

**跟踪整段视频（与训练设置无关）**：下方 `TRACK_FRAMES = None` 表示**不传** `--frames` 参数，sleap-nn 会按**整段视频**推理。这和训练时的「随机预测 20 帧」**无关**——训练时那是验证阶段采样，不会影响推理；推理时只要不传 `--frames` 就是全视频。

**若填了 `"0-999"` 报「非法参数」**：部分 sleap-nn 版本对 `--frames` 的格式要求不同（例如只认逗号分隔 `"0,1,2,3"`）。要**整段视频**请保持 `TRACK_FRAMES = None`；必须限制范围时可在终端运行 `sleap-nn track --help` 查看当前版本的 `--frames` 说明。

**若 .slp 帧数异常少**：(1) 确认输入是 .mp4 且时长正常；(2) 确认 `TRACK_FRAMES` 为 `None`；(3) 用打印出的命令在终端单独跑一次，看日志里的总帧数。

输出路径：`输出根目录 / 模型子目录 / clips子目录 / {模型子目录}_{视频stem}.prediction.slp`  
例如：`D:\sleap_output\top_mouth_375\20251225_K_1_2_clips\top_mouth_375_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.prediction.slp`

**PowerShell 单行（单个 top 视频，可直接粘贴）：**

```powershell
sleap track -i "D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.mp4" -o "D:\sleap_output\top_mouth_375\20251225_K_1_2_clips\top_mouth_375_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.prediction.slp" -n 3 -m "D:\sleap_top_mouth\models\260210_165744.centroid.n=375" -m "D:\sleap_top_mouth\models\260210_181824.centered_instance.n=375" --device gpu
```

In [1]:
# 路径与配置（自动检测项目根）
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents) if (p / "pyproject.toml").exists()), Path.cwd())
SLEAP_OUTPUT_DIR = PROJECT_ROOT / "sleap_output"
QC_OUTPUT_DIR = PROJECT_ROOT  / "sleap_qc"

In [3]:
# 1.1 视频推理：仅填 top 视频列表，自动用 top 模型跟踪 top、用 side 模型跟踪自动找到的 side 视频
# 使用单一进度条显示推理进度，不刷屏
import sys
from pathlib import Path
from tqdm.auto import tqdm

# 确保项目根在 path 中，以便从 scripts.sleap 导入
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.sleap.batch_track import (
    find_side_videos,
    get_model_subdir,
    generate_prediction_slp_path,
    run_sleap_tracking,
)

def _make_progress_callback(pbar):
    """返回用于 run_sleap_tracking 的 progress_callback，只更新同一个进度条"""
    def cb(current, total):
        pbar.total = total
        pbar.n = current
        pbar.refresh()
    return cb

# 仅填写 top 视角视频；会根据路径自动查找 PLAYER2/PLAYER3 或 cam1/cam2 的 side 视频并用 side 模型跟踪
VIDEO_LIST = [
    Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.mp4"),
    Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER1_m1tf_Wrap_387.663-395.277.mp4"),
    Path(r"D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER1_m2tm1_Dash_1209.175-1213.349.mp4"),
]

# Top / Side 模型（与 sleap track -m 顺序一致：centroid, instance）
TOP_MODELS = [
    Path(r"D:\sleap_top_mouth\models\260104_142312.centroid.n=180"),
    Path(r"D:\sleap_top_mouth\models\260104_183212.centered_instance.n=180"),
]
SIDE_MODELS = [
    Path(r"D:\sleap_side_mouth\models\260111_170857.centroid.n=593"),
    Path(r"D:\sleap_side_mouth\models\260112_115936.centered_instance.n=593"),
]

OUTPUT_BASE = Path(r"D:\sleap_output")
MAX_INSTANCES = 3
DEVICE = "gpu"
# 跟踪范围：None = 不传 --frames，整段视频（与训练时「随机 20 帧」无关）；限制范围时格式依 sleap-nn 版本可能为 "0-999" 或 "0,1,2,..."，若报非法参数请保持 None 或查 sleap-nn track --help
TRACK_FRAMES = None
AUTO_TRACK_SIDE = True   # 是否根据 top 视频自动查找并用 side 模型跟踪 side 视频
SKIP_MISSING_SIDE = True # 找不到 side 视频时跳过（False 则报错）

if not VIDEO_LIST:
    print("VIDEO_LIST 为空：请在上方填写 top 视频路径。")
else:
    top_subdir = get_model_subdir(TOP_MODELS)
    side_subdir = get_model_subdir(SIDE_MODELS)
    n_top = len(VIDEO_LIST)
    for idx, top_video in enumerate(VIDEO_LIST, 1):
        if not top_video.exists():
            print(f"跳过（不存在）: {top_video}")
            continue
        print(f"\n{'='*60}\n[ {idx}/{n_top} ] Top: {top_video.name}\n{'='*60}")

        # 1) Top 视频用 top 模型
        out_top = generate_prediction_slp_path(top_video, OUTPUT_BASE, top_subdir)
        print(f"  [Top] -> {out_top.name}")
        pbar_top = tqdm(total=1, desc="  Top 推理", unit="frame")
        try:
            run_sleap_tracking(
                video_path=top_video,
                model_paths=TOP_MODELS,
                output_path=out_top,
                max_instances=MAX_INSTANCES,
                frames=TRACK_FRAMES,
                device=DEVICE,
                progress_callback=_make_progress_callback(pbar_top),
            )
        finally:
            pbar_top.close()

        # 2) 自动查找 side 视频并用 side 模型跟踪
        if AUTO_TRACK_SIDE:
            side1, side2 = find_side_videos(top_video)
            for label, side_video in [("Side1", side1), ("Side2", side2)]:
                if side_video is None:
                    if not SKIP_MISSING_SIDE:
                        raise FileNotFoundError(f"未找到对应 {label} 视频: {top_video}")
                    print(f"  [{label}] 未找到，跳过")
                    continue
                out_side = generate_prediction_slp_path(side_video, OUTPUT_BASE, side_subdir)
                print(f"  [{label}] {side_video.name} -> {out_side.name}")
                pbar_side = tqdm(total=1, desc=f"  {label} 推理", unit="frame")
                try:
                    run_sleap_tracking(
                        video_path=side_video,
                        model_paths=SIDE_MODELS,
                        output_path=out_side,
                        max_instances=MAX_INSTANCES,
                        frames=TRACK_FRAMES,
                        device=DEVICE,
                        progress_callback=_make_progress_callback(pbar_side),
                    )
                finally:
                    pbar_side.close()
    print("\n批量 sleap track 完成。输出根目录:", OUTPUT_BASE)


[ 1/3 ] Top: 20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.mp4
  [Top] -> top_mouth_180_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.prediction.slp


  Top 推理:   0%|          | 0/1 [00:00<?, ?frame/s]


执行跟踪命令:
  sleap track -i D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.mp4 -o D:\sleap_output\top_mouth_180\20251225_K_1_2_clips\top_mouth_180_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.prediction.slp -n 3 -m D:\sleap_top_mouth\models\260104_142312.centroid.n=180 -m D:\sleap_top_mouth\models\260104_183212.centered_instance.n=180 --device gpu
  跟踪范围: 整段视频（未指定 --frames）

2026-02-25 23:28:15 | Started inference at: 2026-02-25 23:28:15.378253
2026-02-25 23:28:15 | sleap-nn 0.1.0 | Python 3.11.14 | PyTorch 2.10.0+cu130 | CUDA 13.0 | 1 GPU(s)
2026-02-25 23:28:15 | Using device: gpu

2026-02-25 23:29:34 | Finished inference at: 2026-02-25 23:29:34.344550
2026-02-25 23:29:34 | Total runtime: 78.96629691123962 secs
2026-02-25 23:29:34 | Predictions output path: D:\sleap_output\top_mouth_180\20251225_K_1_2_clips\top_mouth_180_20251225_K_1_2_FCMC_PLAYER1_m1tf_Courtship_290.729-296.331.prediction.slp
2026-02-25 23:29:34 | Sa

  Side1 推理:   0%|          | 0/1 [00:00<?, ?frame/s]


执行跟踪命令:
  sleap track -i D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.mp4 -o D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.slp -n 3 -m D:\sleap_side_mouth\models\260111_170857.centroid.n=593 -m D:\sleap_side_mouth\models\260112_115936.centered_instance.n=593 --device gpu
  跟踪范围: 整段视频（未指定 --frames）

2026-02-25 23:29:45 | Started inference at: 2026-02-25 23:29:45.420101
2026-02-25 23:29:45 | sleap-nn 0.1.0 | Python 3.11.14 | PyTorch 2.10.0+cu130 | CUDA 13.0 | 1 GPU(s)
2026-02-25 23:29:45 | Using device: gpu

2026-02-25 23:31:38 | Finished inference at: 2026-02-25 23:31:38.573215
2026-02-25 23:31:38 | Total runtime: 113.15311336517334 secs
2026-02-25 23:31:38 | Predictions output path: D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.slp
2026-02-25 23:31:

  Side2 推理:   0%|          | 0/1 [00:00<?, ?frame/s]

... (9 outputs left out)

  Side1 推理:   0%|          | 0/1 [00:00<?, ?frame/s]


执行跟踪命令:
  sleap track -i D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER2_m2tm1_Dash_1209.175-1213.349.mp4 -o D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m2tm1_Dash_1209.175-1213.349.prediction.slp -n 3 -m D:\sleap_side_mouth\models\260111_170857.centroid.n=593 -m D:\sleap_side_mouth\models\260112_115936.centered_instance.n=593 --device gpu
  跟踪范围: 整段视频（未指定 --frames）

2026-02-25 23:42:26 | Started inference at: 2026-02-25 23:42:26.408528
2026-02-25 23:42:26 | sleap-nn 0.1.0 | Python 3.11.14 | PyTorch 2.10.0+cu130 | CUDA 13.0 | 1 GPU(s)
2026-02-25 23:42:26 | Using device: gpu

2026-02-25 23:43:50 | Finished inference at: 2026-02-25 23:43:50.878211
2026-02-25 23:43:50 | Total runtime: 84.46968293190002 secs
2026-02-25 23:43:51 | Predictions output path: D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m2tm1_Dash_1209.175-1213.349.prediction.slp
2026-02-25 23:43:51 | Sa

  Side2 推理:   0%|          | 0/1 [00:00<?, ?frame/s]


执行跟踪命令:
  sleap track -i D:\behavior_clips\20251225_K_1_2_FCMC_clips\20251225_K_1_2_FCMC_PLAYER3_m2tm1_Dash_1209.175-1213.349.mp4 -o D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER3_m2tm1_Dash_1209.175-1213.349.prediction.slp -n 3 -m D:\sleap_side_mouth\models\260111_170857.centroid.n=593 -m D:\sleap_side_mouth\models\260112_115936.centered_instance.n=593 --device gpu
  跟踪范围: 整段视频（未指定 --frames）

2026-02-25 23:44:00 | Started inference at: 2026-02-25 23:44:00.718125
2026-02-25 23:44:00 | sleap-nn 0.1.0 | Python 3.11.14 | PyTorch 2.10.0+cu130 | CUDA 13.0 | 1 GPU(s)
2026-02-25 23:44:00 | Using device: gpu

2026-02-25 23:45:26 | Finished inference at: 2026-02-25 23:45:26.337632
2026-02-25 23:45:26 | Total runtime: 85.61950659751892 secs
2026-02-25 23:45:26 | Predictions output path: D:\sleap_output\side_mouth_593\20251225_K_1_2_clips\side_mouth_593_20251225_K_1_2_FCMC_PLAYER3_m2tm1_Dash_1209.175-1213.349.prediction.slp
2026-02-25 23:45:26 | Sa

## 2. 导出跟踪数据（.slp → CSV / H5）

将 `.slp` 导出为 CSV（含所有 tracks 时使用 `--all-tracks`）或 H5。

**与 overlay 视频一致的轨迹**：本脚本使用 `--all-tracks` 时，会从 .slp 中读取**与 overlay 相同的 track 身份**（Track_0 / Track_1 / Track_2 或 track name），导出的 CSV 与「sleap render」叠加视频中的轨迹一一对应。

In [2]:
# 导出文件夹中所有 .slp 为 CSV（与 overlay 视频同一套轨迹：Track_0/1/2）
# 若使用 D:\sleap_output 请将 slp_dir 设为该路径
import sys
import subprocess
from pathlib import Path

slp_dir = Path(r"D:\sleap_output")  # 或 SLEAP_OUTPUT_DIR；会递归查找子目录中的 .slp
slp_files = sorted(slp_dir.rglob("*.slp")) if slp_dir.exists() else []

if not slp_files:
    print("未在 slp_dir 中找到 .slp 文件。")
    print("命令行等价示例: python scripts/sleap/export_tracks.py --input xxx.slp --format analysis.csv --all-tracks --output <目录>")
else:
    print(f"找到 {len(slp_files)} 个 .slp，将逐个导出为 CSV（与 overlay 一致的 track 身份）。")
    for slp_path in slp_files:
        out_dir = slp_path.parent  # CSV 与 .slp 同目录
        subprocess.run(
            [
                sys.executable,
                str(PROJECT_ROOT / "scripts" / "sleap" / "export_tracks.py"),
                "--input", str(slp_path),
                "--format", "analysis.csv",
                "--all-tracks",
                "--output", str(out_dir),
            ],
            cwd=PROJECT_ROOT,
            check=True,
        )
    print("全部导出完成。CSV 与各 .slp 同目录，根目录:", slp_dir)

找到 11 个 .slp，将逐个导出为 CSV（与 overlay 一致的 track 身份）。
全部导出完成。CSV 与各 .slp 同目录，根目录: D:\sleap_output


## 3. 生成叠加视频（SLEAP 坐标 + 原视频）

对每个 `.slp` 生成「原视频 + SLEAP 关键点/骨架」的叠加视频，输出与 `.slp` 同目录，文件名为 `{slp_stem}.overlay.mp4`。  
优先使用 `sleap render`；若未安装或失败，则用 OpenCV + SLEAP Labels 逐帧绘制。

In [1]:
# 生成 SLEAP 坐标与原视频叠加视频（与导出步骤共用 slp_dir，或单独指定 overlay_slp_dir）
import subprocess
import sys
from pathlib import Path

import numpy as np

overlay_slp_dir = Path(r"D:\sleap_output")  # 与上一步 slp_dir 一致；会递归查找 .slp
slp_files = sorted(overlay_slp_dir.rglob("*.slp")) if overlay_slp_dir.exists() else []

def _load_labels(slp_path):
    """加载 .slp，兼容 sleap / sleap_io。"""
    try:
        from sleap.io.dataset import Labels
        return Labels.load_file(str(slp_path))
    except Exception:
        try:
            import sleap
            return sleap.load_file(str(slp_path))
        except Exception:
            try:
                import sleap_io as sio
                return sio.load_slp(str(slp_path))
            except Exception as e:
                raise RuntimeError(f"无法加载 .slp: {e}") from e

def _get_skeleton_edges(labels):
    """返回 (node_names, edges)。edges 为 [(i,j)] 索引对或 [(name_i, name_j)] 名称对。"""
    sk = getattr(labels, "skeleton", None)
    if sk is None and hasattr(labels, "videos") and len(labels.videos) > 0:
        sk = getattr(labels.videos[0], "skeleton", None)
    if sk is None:
        return [], []
    if hasattr(sk, "node_names"):
        names = [str(n) for n in sk.node_names]
    elif hasattr(sk, "nodes"):
        names = [str(n) for n in sk.nodes]
    else:
        names = []
    edges = []
    if hasattr(sk, "edge_inds") and sk.edge_inds is not None:
        for e in np.asarray(sk.edge_inds):
            edges.append((int(e[0]), int(e[1])))
    elif hasattr(sk, "edges") and sk.edges is not None:
        for e in sk.edges:
            if isinstance(e[0], (int, np.integer)):
                edges.append((int(e[0]), int(e[1])))
            else:
                edges.append((str(e[0]), str(e[1])))
    return names, edges

def _instance_points(inst, labels, lf):
    """从 instance 得到 node_name -> (x, y) 的 dict；与 export_tracks 解析方式一致。"""
    pts = getattr(inst, "points", None)
    if pts is None:
        return {}
    out = {}
    if hasattr(pts, "items"):
        for node, p in pts.items():
            if getattr(p, "score", 1.0) < 0.01:
                continue
            out[str(node)] = (float(getattr(p, "x", p[0])), float(getattr(p, "y", p[1])))
    else:
        sk = getattr(labels, "skeleton", None)
        if sk is None and hasattr(lf, "video"):
            sk = getattr(lf.video, "skeleton", None)
        node_names = []
        if sk is not None:
            node_names = getattr(sk, "node_names", None) or getattr(sk, "nodes", [])
            node_names = [str(n) for n in node_names]
        points_array = getattr(inst, "numpy", lambda: None)() if hasattr(inst, "numpy") else pts
        if isinstance(points_array, np.ndarray) and len(points_array.shape) >= 2:
            for i in range(points_array.shape[0]):
                name = node_names[i] if i < len(node_names) else f"node_{i}"
                score = float(points_array[i, 2]) if points_array.shape[1] > 2 else 1.0
                if score < 0.01:
                    continue
                out[name] = (float(points_array[i, 0]), float(points_array[i, 1]))
    return out

def _render_overlay_python(slp_path: Path, out_path: Path) -> None:
    """用 OpenCV 将 .slp 的预测画到原视频上并保存。"""
    try:
        import cv2
    except ImportError:
        raise RuntimeError("Python 叠加需要 opencv-python，请安装: pip install opencv-python") from None

    labels = _load_labels(slp_path)
    video_path = None
    if hasattr(labels, "videos") and len(labels.videos) > 0:
        v0 = labels.videos[0]
        fname = getattr(v0, "filename", getattr(v0, "path", None)) or str(v0)
        video_path = Path(fname)
        if not video_path.is_absolute():
            for base in [slp_path.parent, overlay_slp_dir]:
                cand = base / video_path.name
                if cand.exists():
                    video_path = cand
                    break
            else:
                video_path = (slp_path.parent / video_path.name).resolve()
    if video_path is None or not video_path.exists():
        # 尝试同目录下同名 .mp4
        video_path = slp_path.parent / f"{slp_path.stem.replace('.prediction', '')}.mp4"
        if not video_path.exists():
            stem = slp_path.stem
            for suffix in [".mp4", ".avi", ".mov"]:
                for prefix in ["", "top_mouth_375_", "top_mouth_180_"]:
                    cand = slp_path.parent / f"{prefix}{stem.replace('.prediction', '').replace(prefix, '')}{suffix}".replace("..", ".")
                    if cand.exists():
                        video_path = cand
                        break
            if not video_path.exists():
                raise FileNotFoundError(f"未找到与 .slp 对应的视频，请确保 .slp 内记录的路径正确或把视频放在 .slp 同目录: {slp_path}")

    node_names, edges = _get_skeleton_edges(labels)
    # 按帧整理 instances: frame_idx -> [(points_dict, track_id), ...]
    by_frame = {}
    for lf in labels.labeled_frames:
        fidx = int(lf.frame_idx)
        for inst in lf.instances:
            pts = _instance_points(inst, labels, lf)
            if not pts:
                continue
            tid = getattr(inst.track, "name", None) or getattr(inst.track, "track_id", id(inst))
            by_frame.setdefault(fidx, []).append((pts, tid))

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))

    track_colors = {}  # track_id -> (B, G, R)
    palette = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        data = by_frame.get(frame_idx, [])
        for pts_dict, tid in data:
            if tid not in track_colors:
                track_colors[tid] = palette[len(track_colors) % len(palette)]
            color = track_colors[tid]
            # 画关节点
            for name, (x, y) in pts_dict.items():
                ix, iy = int(round(x)), int(round(y))
                if 0 <= ix < w and 0 <= iy < h:
                    cv2.circle(frame, (ix, iy), 3, color, -1)
            # 画骨架边（支持 (i,j) 索引或 (name_i, name_j) 名称对）
            for e0, e1 in edges:
                if isinstance(e0, int) and isinstance(e1, int):
                    if e0 >= len(node_names) or e1 >= len(node_names):
                        continue
                    ni, nj = node_names[e0], node_names[e1]
                else:
                    ni, nj = str(e0), str(e1)
                if ni in pts_dict and nj in pts_dict:
                    p1 = (int(round(pts_dict[ni][0])), int(round(pts_dict[ni][1])))
                    p2 = (int(round(pts_dict[nj][0])), int(round(pts_dict[nj][1])))
                    cv2.line(frame, p1, p2, color, 1)
        writer.write(frame)
        frame_idx += 1
    cap.release()
    writer.release()

if not slp_files:
    print("未在 overlay_slp_dir 中找到 .slp 文件。")
else:
    print(f"找到 {len(slp_files)} 个 .slp，将逐个生成叠加视频。")
    for slp_path in slp_files:
        out_video = slp_path.parent / f"{slp_path.stem}.overlay.mp4"
        try:
            subprocess.run(
                ["sleap", "render", str(slp_path), "-o", str(out_video)],
                cwd=str(slp_path.parent),
                check=True,
                capture_output=True,
            )
            print(f"  [sleap render] {slp_path.name} -> {out_video.name}")
        except (FileNotFoundError, subprocess.CalledProcessError):
            try:
                _render_overlay_python(slp_path, out_video)
                print(f"  [Python 叠加] {slp_path.name} -> {out_video.name}")
            except Exception as e:
                print(f"  跳过 {slp_path.name}: {e}")
    print("叠加视频生成完成。输出与各 .slp 同目录。")

找到 11 个 .slp，将逐个生成叠加视频。
  [sleap render] side_mouth_437_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.slp -> side_mouth_437_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.overlay.mp4
  [sleap render] side_mouth_437_20251225_K_1_2_FCMC_PLAYER3_m1tf_Courtship_290.729-296.331.prediction.slp -> side_mouth_437_20251225_K_1_2_FCMC_PLAYER3_m1tf_Courtship_290.729-296.331.prediction.overlay.mp4
  [sleap render] side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.slp -> side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Courtship_290.729-296.331.prediction.overlay.mp4
  [sleap render] side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Wrap_387.663-395.277.prediction.slp -> side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m1tf_Wrap_387.663-395.277.prediction.overlay.mp4
  [sleap render] side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m2tm1_Dash_1209.175-1213.349.prediction.slp -> side_mouth_593_20251225_K_1_2_FCMC_PLAYER2_m2tm1_Dash_1